In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
 Augment a Parameter-Free "BEST_CONFIGS on TEST" results workbook with
 analysis sheets.
================================================================================

Companion to augment_pdep_report.py, adapted for the parameter-free TEST
evaluation produced by evaluate_paramfree_best_configs_on_test.py -- a
"Results" sheet with columns:

    Fold | System | Rule | EER (%) | Rank-1 @ FAR=1% | N_test

IMPORTANT DIFFERENCE FROM THE P-DEPENDENT VERSION
--------------------------------------------------
That source file contains ONLY the single, already-selected winning rule per
system (chosen on TRAIN, with no p to sweep), evaluated on the 5 held-out TEST
folds. There is no rule sweep and no p axis in this data -- unlike the TRAIN
sweep workbook, which has 10 rules x 21 p-values per system.

Sheets that depended on comparing many (rule, p) candidates -- P-Sensitivity,
"best EER vs p", "EER vs p for the winning rule" -- have no equivalent here,
because there is nothing to sweep or compare against. Rather than force
degenerate, mostly-empty versions of those sheets, this script REPLACES them
with the sheets that are actually meaningful for this data: a fold-by-fold
breakdown and a fold-consistency (variability) analysis. This substitution is
explained explicitly in the Methodology sheet, not left implicit.

Sheets added (or refreshed if already present):
  1. Methodology                - detailed description of data and methods
  2. Best_By_EER                - per-system TEST summary (mean/std/min/max/CI95)
  3. Best_by_R1FAR1              - identical rows to (2) here; see Methodology for why
  4. EER_Sensitivity_Analysis    - System x Rule mean EER (degenerate: 1 rule/system)
  5. R1FAR1_Sensitivity_Analysis - System x Rule mean Rank-1 (degenerate: 1 rule/system)
  6. Fold_Consistency_Analysis   - REPLACES P-Sensitivity: EER/Rank-1 std across folds
  7. EER_by_Fold                 - REPLACES BestEER_vs_P: Fold x System EER matrix
  8. Rank1_by_Fold                - REPLACES EER_vs_P_WinningRule: Fold x System Rank-1 matrix

USAGE
-----
    python augment_paramfree_test_report.py ParamFree_BestConfigs_TEST_seed62.xlsx
================================================================================
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from scipy import stats
from openpyxl import load_workbook
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.worksheet import Worksheet


# ==============================================================================
#  CONFIGURATION
# ==============================================================================

RESULTS_XLSX_FILE = "./Results/ParamFree_BestConfigs_TEST_seed62.xlsx"
RESULTS_SHEET_NAME = "Results"
# This workbook's writer used 3 text rows (title, subtitle, configs) + 1 blank
# + header, so the header sits on row 5 (0-indexed row 4). Different from the
# p-dependent workbook, which has no "configs used" line and headers on row 4.
RESULTS_HEADER_ROW = 3
EER_LABEL = "EER (%)"
FAR_LABEL = "Rank-1 @ FAR=1%"

THEME_COLOR = "6A1B9A"   # same purple family as the p-dependent TEST script
FONT_NAME = "Arial"
CONFIDENCE_LEVEL = 0.95


# ==============================================================================
#  STEP 0 -- LOAD THE SOURCE DATA
# ==============================================================================

def load_results(filepath: Path) -> pd.DataFrame:
    """Read the raw (Fold, System, Rule, EER, Rank-1, N_test) table."""
    frame = pd.read_excel(filepath, sheet_name=RESULTS_SHEET_NAME,
                          header=RESULTS_HEADER_ROW)
    required = {"Fold", "System", "Rule", EER_LABEL, FAR_LABEL}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(
            f"Results sheet is missing expected columns: {sorted(missing)}. "
            f"Found columns: {list(frame.columns)}"
        )
    return frame


def system_order(frame: pd.DataFrame) -> List[str]:
    canonical = [f"S{i}" for i in range(1, 9)]
    present = set(frame["System"])
    return [s for s in canonical if s in present]


def rules_used(frame: pd.DataFrame) -> List[str]:
    """
    Every rule that appears anywhere in the data, in first-seen order. In this
    TEST file there is exactly one rule per system, so this list typically has
    as many entries as there are distinct winning rules (<= number of systems).
    """
    seen: List[str] = []
    for rule in frame["Rule"]:
        if rule not in seen:
            seen.append(rule)
    return seen


# ==============================================================================
#  STEP 1 -- STATISTICS HELPERS
# ==============================================================================

def ci95_half_width(values: pd.Series) -> float:
    """95% CI half-width via the t-distribution, for a small-sample mean."""
    n = values.count()
    if n < 2:
        return 0.0
    alpha = 1.0 - CONFIDENCE_LEVEL
    t_crit = stats.t.ppf(1.0 - alpha / 2.0, df=n - 1)
    return float(t_crit * values.std(ddof=1) / np.sqrt(n))


def describe(values: pd.Series, prefix: str) -> dict:
    """mean / std / min / max / CI95 block for one metric."""
    return {
        f"{prefix}_mean": float(values.mean()),
        f"{prefix}_std":  float(values.std(ddof=1)) if values.count() > 1 else 0.0,
        f"{prefix}_min":  float(values.min()),
        f"{prefix}_max":  float(values.max()),
        f"{prefix}_CI95": ci95_half_width(values),
    }


# ==============================================================================
#  STEP 2 -- BUILD EACH TABLE
# ==============================================================================

def build_summary_by_system(frame: pd.DataFrame, systems: List[str]) -> pd.DataFrame:
    """
    Sheets 2 & 3: per system, full statistics of its (single, fixed) winning
    rule's TEST performance across the 5 folds. Since only one rule was ever
    evaluated per system in this file, "Best_By_EER" and "Best_by_R1FAR1" are
    the SAME table here -- there is no second candidate to select differently
    against. Both sheets are still written, for structural consistency with
    the TRAIN-side workbook, but they are identical by construction.
    """
    rows = []
    for system in systems:
        subset = frame[frame.System == system]
        rule = subset["Rule"].iloc[0]  # exactly one rule per system in this file
        row = {"System": system, "Rule": rule}
        row.update(describe(subset[EER_LABEL], "EER"))
        row.update({
            "Rank1_mean": subset[FAR_LABEL].mean(),
            "Rank1_std":  subset[FAR_LABEL].std(ddof=1) if len(subset) > 1 else 0.0,
        })
        row.update(describe(subset[FAR_LABEL], "Rank1FAR1"))
        rows.append(row)
    result = pd.DataFrame(rows)

    # Reorder / relabel to mirror the requested column set, with stats appended.
    ordered = pd.DataFrame({
        "System": result["System"],
        "Rule": result["Rule"],
        "EER (%)": result["EER_mean"].round(4),
        "Rank-1 (%)": result["Rank1FAR1_mean"].round(4),
        "Rank-1 @ FAR=1%": result["Rank1FAR1_mean"].round(4),
        "EER_std": result["EER_std"].round(4),
        "EER_min": result["EER_min"].round(4),
        "EER_max": result["EER_max"].round(4),
        "EER_CI95": result["EER_CI95"].round(4),
        "Rank1FAR1_std": result["Rank1FAR1_std"].round(4),
        "Rank1FAR1_min": result["Rank1FAR1_min"].round(4),
        "Rank1FAR1_max": result["Rank1FAR1_max"].round(4),
        "Rank1FAR1_CI95": result["Rank1FAR1_CI95"].round(4),
    })
    return ordered


def build_sensitivity(frame: pd.DataFrame, systems: List[str], rules: List[str],
                      value_col: str) -> pd.DataFrame:
    """
    Sheets 4 & 5: System x Rule mean of `value_col`. In this TEST file there
    is exactly one rule per system, so most cells are empty by construction
    (that system's rule was simply never tested against the others here) --
    this is the honest reflection of what this file actually contains, not a
    computation error. It is kept for structural parity with the TRAIN-side
    workbook, where every rule WAS tested against every system.
    """
    pivot = (
        frame.groupby(["System", "Rule"])[value_col].mean()
        .unstack("Rule").reindex(index=systems, columns=rules)
    )
    pivot.insert(0, "System", pivot.index)
    return pivot.reset_index(drop=True)


def build_fold_consistency(frame: pd.DataFrame, systems: List[str]) -> pd.DataFrame:
    """
    Sheet 6 (replaces P_Sensitivity_Analysis, which has no meaning without a
    p parameter): for each system, how much does performance vary across the
    5 TEST folds? A parameter-free rule cannot be "sensitive to p" -- the
    equivalent question here is "how sensitive is it to which subjects ended
    up in which fold", answered by the fold-to-fold standard deviation.
    """
    rows = []
    for system in systems:
        subset = frame[frame.System == system]
        rows.append({
            "System": system,
            "Rule": subset["Rule"].iloc[0],
            "EER_std_across_folds": round(subset[EER_LABEL].std(ddof=1), 4)
                                    if len(subset) > 1 else 0.0,
            "EER_range (max-min)": round(subset[EER_LABEL].max() - subset[EER_LABEL].min(), 4),
            "Rank1_std_across_folds": round(subset[FAR_LABEL].std(ddof=1), 4)
                                      if len(subset) > 1 else 0.0,
            "Rank1_range (max-min)": round(subset[FAR_LABEL].max() - subset[FAR_LABEL].min(), 4),
        })
    return pd.DataFrame(rows).sort_values("EER_std_across_folds", ascending=False).reset_index(drop=True)


def build_metric_by_fold(frame: pd.DataFrame, systems: List[str],
                         value_col: str) -> pd.DataFrame:
    """
    Sheets 7 & 8 (replace the p-based sweep sheets, which have no equivalent
    here): Fold x System matrix of the raw metric value, so fold-to-fold
    patterns are visible directly rather than only summarized as a std.
    """

   
    pivot = frame.pivot(index="Fold", columns="System", values=value_col).reindex(columns=systems)
    pivot.insert(0, "Fold", pivot.index)
    return pivot.reset_index(drop=True)


# ==============================================================================
#  STEP 3 -- METHODOLOGY TEXT
# ==============================================================================

def methodology_lines(frame: pd.DataFrame, systems: List[str]) -> List[tuple]:
    n_folds = int(frame["Fold"].nunique())
    configs = frame.drop_duplicates("System")[["System", "Rule"]]
    configs_text = "; ".join(f"{r.System}={r.Rule}" for r in configs.itertuples(index=False))

    return [
        ("title", "Methodology"),
        ("note",  "How every derived sheet in this workbook was computed from the Results sheet."),
        ("blank", ""),
        ("head",  "SOURCE DATA"),
        ("body",  f"The 'Results' sheet holds {len(frame)} raw rows: {n_folds} cross-validation "
                  f"folds x {len(systems)} fusion systems. Unlike the p-dependent TRAIN workbook, "
                  "this file contains NO sweep -- each system was fused with exactly ONE fixed, "
                  "already-selected parameter-free rule (there is no p for this rule family), "
                  "and evaluated on that fold's held-out 20% TEST subjects."),
        ("body",  f"Winning rules used (selected on TRAIN, before this file was created): {configs_text}."),
        ("body",  "Normalization was fit on each fold's TRAIN submatrix only and applied unchanged "
                  "to that fold's TEST submatrix -- the test subjects never influenced their own "
                  "normalization scale, and the rule choice never touched test data either."),
        ("blank", ""),
        ("head",  "WHY SOME SHEETS DIFFER FROM THE TRAIN-SWEEP WORKBOOK"),
        ("body",  "The TRAIN-side workbook (augment_pdep_report.py) has sheets built by comparing "
                  "many (rule, p) candidates against each other: P-Sensitivity, best-EER-vs-p, "
                  "EER-vs-p-for-the-winning-rule. None of those comparisons are possible here, "
                  "because this file only ever tested ONE configuration per system -- the winner "
                  "was already chosen before this evaluation ran. Building those same sheets here "
                  "would produce mostly-empty tables with a single filled cell per row."),
        ("body",  "Instead, this workbook substitutes the sheets that ARE meaningful for a "
                  "single-fixed-configuration TEST evaluation: a fold-consistency analysis (how "
                  "much performance varies across the 5 folds, replacing the idea of 'sensitivity "
                  "to p' with 'sensitivity to which subjects landed in which fold'), and direct "
                  "fold-by-fold breakdowns of both metrics."),
        ("blank", ""),
        ("head",  "SHEET-BY-SHEET DEFINITIONS"),
        ("body",  "Best_By_EER / Best_by_R1FAR1: full TEST statistics (mean, std, min, max, 95% CI) "
                  "per system, for its single fixed winning rule. The two sheets are IDENTICAL here "
                  "-- kept separate only for structural parity with the TRAIN workbook, since there "
                  "is no second candidate configuration to select differently against."),
        ("body",  "EER_Sensitivity_Analysis / R1FAR1_Sensitivity_Analysis: System x Rule pivot. "
                  "Mostly empty by construction, since each system in this file has exactly one "
                  "rule; the single filled cell per row is that system's mean TEST value."),
        ("body",  "Fold_Consistency_Analysis: standard deviation and range (max-min) of EER and "
                  "Rank-1 across the 5 TEST folds, per system, sorted from least to most stable. "
                  "A large std/range means that system's performance is sensitive to exactly which "
                  "subjects ended up in the held-out fold -- worth flagging even though the rule "
                  "itself never changes."),
        ("body",  "EER_by_Fold / Rank1_by_Fold: the raw Fold x System matrix, so the individual "
                  "numbers behind every mean in Best_By_EER can be inspected directly."),
        ("blank", ""),
        ("head",  "ON THE 'Rank-1 (%)' AND 'Rank-1 @ FAR=1%' COLUMNS"),
        ("body",  "As in the p-dependent workbook, the source data contains a single identification "
                  "metric (Rank-1 at 1% FAR). Both columns report the same value; this is stated "
                  "explicitly rather than implying two different metrics were computed."),
    ]


# ==============================================================================
#  STEP 4 -- EXCEL WRITING HELPERS (shared style with the p-dependent version)
# ==============================================================================

def _styles():
    thin = Side(style="thin", color="D9D9D9")
    return {
        "header_fill": PatternFill("solid", fgColor=THEME_COLOR),
        "header_font": Font(name=FONT_NAME, bold=True, color="FFFFFF", size=10),
        "title_font":  Font(name=FONT_NAME, bold=True, size=13, color=THEME_COLOR),
        "note_font":   Font(name=FONT_NAME, italic=True, size=9, color="555555"),
        "cell_font":   Font(name=FONT_NAME, size=10),
        "border":      Border(left=thin, right=thin, top=thin, bottom=thin),
    }


def _autofit(worksheet: Worksheet, n_columns: int, header_row: int, n_rows: int,
            minimum: int = 9, maximum: int = 32) -> None:
    for col_idx in range(1, n_columns + 1):
        letter = get_column_letter(col_idx)
        longest = max(
            (len(str(worksheet.cell(row=r, column=col_idx).value))
             for r in range(header_row, header_row + n_rows + 1)
             if worksheet.cell(row=r, column=col_idx).value is not None),
            default=minimum,
        )
        worksheet.column_dimensions[letter].width = min(max(longest + 2, minimum), maximum)


def _write_table(worksheet: Worksheet, frame: pd.DataFrame, styles: dict,
                 title: str, subtitle: str = "", percent_cols: tuple = ()) -> int:
    worksheet.cell(row=1, column=1, value=title).font = styles["title_font"]
    row = 2
    if subtitle:
        worksheet.cell(row=row, column=1, value=subtitle).font = styles["note_font"]
        row += 1
    row += 1
    header_row = row

    for col_idx, column in enumerate(frame.columns, start=1):
        cell = worksheet.cell(row=header_row, column=col_idx, value=str(column))
        cell.fill = styles["header_fill"]
        cell.font = styles["header_font"]
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = styles["border"]

    for offset, record in enumerate(frame.itertuples(index=False), start=1):
        for col_idx, value in enumerate(record, start=1):
            if isinstance(value, (np.integer,)):
                value = int(value)
            elif isinstance(value, (np.floating,)):
                value = None if np.isnan(value) else float(value)
            cell = worksheet.cell(row=header_row + offset, column=col_idx, value=value)
            cell.font = styles["cell_font"]
            cell.border = styles["border"]
            column = frame.columns[col_idx - 1]
            if column in percent_cols:
                cell.number_format = "0.0000"

    worksheet.freeze_panes = worksheet.cell(row=header_row + 1, column=1)
    _autofit(worksheet, len(frame.columns), header_row, len(frame))
    return header_row


def _write_methodology(worksheet: Worksheet, lines: List[tuple], styles: dict) -> None:
    worksheet.column_dimensions["A"].width = 108
    fonts = {
        "title": styles["title_font"],
        "head":  Font(name=FONT_NAME, bold=True, size=11),
        "note":  styles["note_font"],
        "body":  styles["cell_font"],
        "blank": styles["cell_font"],
    }
    for row, (kind, text) in enumerate(lines, start=1):
        cell = worksheet.cell(row=row, column=1, value=text)
        cell.font = fonts[kind]
        cell.alignment = Alignment(wrap_text=False, vertical="top")


def _replace_sheet(workbook, name: str):
    if name in workbook.sheetnames:
        del workbook[name]
    return workbook.create_sheet(name[:31])


# ==============================================================================
#  MAIN ENTRY POINT
# ==============================================================================

def augment_workbook(filepath: str) -> Path:
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Workbook not found: {path.resolve()}")

    frame = load_results(path)
    systems = system_order(frame)
    rules = rules_used(frame)

    summary = build_summary_by_system(frame, systems)
    eer_sensitivity = build_sensitivity(frame, systems, rules, EER_LABEL)
    rank1_sensitivity = build_sensitivity(frame, systems, rules, FAR_LABEL)
    fold_consistency = build_fold_consistency(frame, systems)
    #eer_by_fold = build_metric_by_fold(frame, systems, EER_LABEL)
    #rank1_by_fold = build_metric_by_fold(frame, systems, FAR_LABEL)

    workbook = load_workbook(path)
    styles = _styles()

    # 1. Methodology
    ws = _replace_sheet(workbook, "Methodology")
    _write_methodology(ws, methodology_lines(frame, systems), styles)

    # 2. Best_By_EER
    ws = _replace_sheet(workbook, "Best_By_EER")
    _write_table(
        ws, summary, styles,
        title="Per-System TEST Summary \u2014 fixed winning rule, 5-fold statistics",
        subtitle="Rule was selected on TRAIN (not shown here); this sheet reports its honest TEST performance.",
        percent_cols=("EER (%)", "Rank-1 (%)", "Rank-1 @ FAR=1%", "EER_std", "EER_min", "EER_max",
                     "EER_CI95", "Rank1FAR1_std", "Rank1FAR1_min", "Rank1FAR1_max", "Rank1FAR1_CI95"),
    )

    # 3. Best_by_R1FAR1 (identical content -- see Methodology for why)
    ws = _replace_sheet(workbook, "Best_by_R1FAR1")
    _write_table(
        ws, summary, styles,
        title="Per-System TEST Summary \u2014 identical to Best_By_EER (only one rule tested per system)",
        subtitle="No independent Rank-1-optimal selection is possible: this file tested a single fixed rule per system.",
        percent_cols=("EER (%)", "Rank-1 (%)", "Rank-1 @ FAR=1%", "EER_std", "EER_min", "EER_max",
                     "EER_CI95", "Rank1FAR1_std", "Rank1FAR1_min", "Rank1FAR1_max", "Rank1FAR1_CI95"),
    )

    # 4. EER_Sensitivity_Analysis
    ws = _replace_sheet(workbook, "EER_Sensitivity_Analysis")
    hr = _write_table(
        ws, eer_sensitivity, styles,
        title="EER Sensitivity Analysis \u2014 mean EER (%) per System x Rule",
        subtitle="Mostly empty by construction: each system in this file was tested with exactly one rule.",
        percent_cols=tuple(rules),
    )
    if len(rules) > 1:
        last_col = get_column_letter(len(eer_sensitivity.columns))
        ws.conditional_formatting.add(
            f"B{hr+1}:{last_col}{hr+len(eer_sensitivity)}",
            ColorScaleRule(start_type="min", start_color="63BE7B", end_type="max", end_color="F8696B"),
        )

    # 5. R1FAR1_Sensitivity_Analysis
    ws = _replace_sheet(workbook, "R1FAR1_Sensitivity_Analysis")
    hr = _write_table(
        ws, rank1_sensitivity, styles,
        title="Rank-1 @ FAR=1% Sensitivity Analysis \u2014 mean per System x Rule",
        subtitle="Mostly empty by construction: each system in this file was tested with exactly one rule.",
        percent_cols=tuple(rules),
    )
    if len(rules) > 1:
        last_col = get_column_letter(len(rank1_sensitivity.columns))
        ws.conditional_formatting.add(
            f"B{hr+1}:{last_col}{hr+len(rank1_sensitivity)}",
            ColorScaleRule(start_type="min", start_color="F8696B", end_type="max", end_color="63BE7B"),
        )

    # 6. Fold_Consistency_Analysis (replaces P_Sensitivity_Analysis)
    ws = _replace_sheet(workbook, "Fold_Consistency_Analysis")
    _write_table(
        ws, fold_consistency, styles,
        title="Fold Consistency Analysis \u2014 replaces P-Sensitivity (no p exists for parameter-free rules)",
        subtitle="Std and range of EER/Rank-1 across the 5 TEST folds; sorted least to most stable (by EER std).",
        percent_cols=("EER_std_across_folds", "EER_range (max-min)",
                     "Rank1_std_across_folds", "Rank1_range (max-min)"),
    )

    # 7. EER_by_Fold (replaces BestEER_vs_P_AllRules)
    #ws = _replace_sheet(workbook, "EER_by_Fold")
    """
    _write_table(
        ws, eer_by_fold, styles,
        title="EER (%) by Fold \u2014 replaces BestEER_vs_P (no p or rule sweep in this file)",
        subtitle="Raw Fold x System matrix; the individual numbers behind each Best_By_EER mean.",
        percent_cols=tuple(systems),
    )
    """

    # 8. Rank1_by_Fold (replaces EER_vs_P_WinningRule)
    #ws = _replace_sheet(workbook, "Rank1_by_Fold")
    """
    _write_table(
        ws, rank1_by_fold, styles,
        title="Rank-1 @ FAR=1% by Fold \u2014 replaces EER_vs_P_WinningRule (no p sweep in this file)",
        subtitle="Raw Fold x System matrix; the individual numbers behind each Best_By_EER mean.",
        percent_cols=tuple(systems),
    )
    """

    desired_order = [RESULTS_SHEET_NAME, "Methodology", "Best_By_EER", "Best_by_R1FAR1",
                     "EER_Sensitivity_Analysis", "R1FAR1_Sensitivity_Analysis",
                     "Fold_Consistency_Analysis", "EER_by_Fold", "Rank1_by_Fold"]
    remaining = [s for s in workbook.sheetnames if s not in desired_order]
    workbook._sheets = [workbook[s] for s in desired_order if s in workbook.sheetnames] \
                       + [workbook[s] for s in remaining]

    workbook.save(path)
    return path


if __name__ == "__main__":
    
    filepath = RESULTS_XLSX_FILE
    print(f"Augmenting workbook: {filepath}")
    output_path = augment_workbook(filepath)
    print("Done. Sheets added/updated: Methodology, Best_By_EER, Best_by_R1FAR1, "
          "EER_Sensitivity_Analysis, R1FAR1_Sensitivity_Analysis, "
          "Fold_Consistency_Analysis, EER_by_Fold, Rank1_by_Fold")
    print(f"Saved to: {output_path}")

    

Augmenting workbook: ./Results/ParamFree_BestConfigs_TEST_seed62.xlsx
Done. Sheets added/updated: Methodology, Best_By_EER, Best_by_R1FAR1, EER_Sensitivity_Analysis, R1FAR1_Sensitivity_Analysis, Fold_Consistency_Analysis, EER_by_Fold, Rank1_by_Fold
Saved to: Results\ParamFree_BestConfigs_TEST_seed62.xlsx
